[Reference](https://levelup.gitconnected.com/building-a-self-improvement-agent-with-langgraph-reflection-vs-reflexion-1d1abcc5865d)

# Building Reflection Agent


```
# Setting up the Environment
# .env file
ANTHROPIC_API_KEY="your-anthropic-api-key"
# LANGCHAIN_API_KEY="your-langchain-api-key"  # optional
# LANGCHAIN_TRACING_V2=True                   # optional
# LANGCHAIN_PROJECT="multi-agent-swarm"       # optional
```

In [1]:
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv

load_dotenv()
load_dotenv(dotenv_path="../.env", override=True) # mention the .env path

# Initialize Anthropic model
llm = ChatAnthropic(
                      model="claude-3-7-sonnet-latest",  # Claude model ID
                      temperature=0,
                      # max_tokens=1024
 )

# Generator
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

post_creation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert LinkedIn content creator tasked with crafting compelling, professional, and high-performing LinkedIn posts. "
            "Create the most effective LinkedIn post possible based on the user's requirements. "
            "If the user provides feedback or suggestions, respond with an improved version that incorporates their input while enhancing overall quality and engagement.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

# Example LinkedIn post creation session
generated_post = ""
post_request = HumanMessage(
    content="Create a LinkedIn post on AI tools for developers under 200 words."
)

print("=== INITIAL LINKEDIN POST ===")
for chunk in linkedin_post_generator.stream({"messages": [post_request]}):
    print(chunk.content, end="")
    generated_post += chunk.content

print("\n" + "="*60 + "\n")

# Reflect

# SOCIAL MEDIA STRATEGIST REFLECTION
social_media_critique_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a LinkedIn content strategist and thought leadership expert. Analyze the given LinkedIn post and provide a comprehensive critique focusing on:

        **Content Quality & Professionalism:**
        - Overall quality, tone clarity, and LinkedIn best practices alignment
        - Structure, readability, and professional credibility building
        - Industry relevance and audience targeting

        **Engagement & Algorithm Optimization:**
        - Hook effectiveness and storytelling quality
        - Engagement potential (likes, comments, shares)
        - LinkedIn algorithm optimization factors
        - Word count and formatting effectiveness

        **Technical Elements:**
        - Hashtag relevance, reach, and strategic placement
        - Call-to-action strength and clarity
        - Use of formatting (line breaks, bullet points, mentions)

        Provide specific, actionable feedback that includes:
        - Key strengths and improvement areas
        - Concrete suggestions for enhancing engagement and professionalism
        - Practical recommendations for the next revision

        Keep your critique constructive and focused on measurable improvements, prioritizing actionable insights that will guide the post's revision and lead to tangible content enhancements."""
    ),
    MessagesPlaceholder(variable_name="messages")
])

social_media_critic = social_media_critique_prompt | llm

print("=== SOCIAL MEDIA STRATEGIST FEEDBACK ===")
feedback_result = ""
for chunk in social_media_critic.stream({"messages": [post_request, HumanMessage(content=generated_post)]}):
    print(chunk.content, end="")
    feedback_result += chunk.content

print("\n" + "="*60 + "\n")

# Repeat

print("=== REFINED LINKEDIN POST ===")
for chunk in linkedin_post_generator.stream(
    {"messages": [post_request, AIMessage(content=generated_post), HumanMessage(content=feedback_result)]}
):
    print(chunk.content, end="")

print("\n" + "="*60 + "\n")

# Workflow Orchestration with LangGraph

from typing import Annotated, List, Sequence
from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from typing_extensions import TypedDict

class ContentState(TypedDict):
    messages: Annotated[list, add_messages]

async def post_creation_node(state: ContentState) -> ContentState:
    """Generate or improve LinkedIn post based on current state."""
    return {"messages": [await linkedin_post_generator.ainvoke(state["messages"])]}

async def social_critique_node(state: ContentState) -> ContentState:
    """Provide social media strategy feedback for the LinkedIn post."""
    # Transform message types for the strategist
    message_role_map = {"ai": HumanMessage, "human": AIMessage}

    # Keep the original request and transform subsequent messages
    transformed_messages = [state["messages"][0]] + [
        message_role_map[msg.type](content=msg.content) for msg in state["messages"][1:]
    ]

    strategy_feedback = await social_media_critic.ainvoke(transformed_messages)

    # Return feedback as human input for the post generator
    return {"messages": [HumanMessage(content=strategy_feedback.content)]}

def should_continue_refining(state: ContentState):
    """Determine whether to continue the creation-feedback cycle."""
    if len(state["messages"]) > 6:
        # End after 3 complete creation-feedback cycles
        return END
    return "social_critique"

# Build the workflow graph
content_workflow_builder = StateGraph(ContentState)
content_workflow_builder.add_node("create_post", post_creation_node)
content_workflow_builder.add_node("social_critique", social_critique_node)

# Define workflow edges
content_workflow_builder.add_edge(START, "create_post")
content_workflow_builder.add_conditional_edges("create_post", should_continue_refining)
content_workflow_builder.add_edge("social_critique", "create_post")

# Add conversation memory
content_memory = InMemorySaver()
linkedin_workflow = content_workflow_builder.compile(checkpointer=content_memory)

from IPython.display import Image, display

# Show the agent
display(Image(linkedin_workflow.get_graph().draw_png()))

# Testing

session_config = {"configurable": {"thread_id": "user1"}}

content_brief = HumanMessage(
    content="Create a LinkedIn post on AI tools for developers under 180 words."
)

async for workflow_event in linkedin_workflow.astream(
    {"messages": [content_brief]},
    session_config,
):
    print("Workflow Step:", workflow_event)
    print("-" * 50)

# Get final state
final_state = linkedin_workflow.get_state(session_config)
print("Total messages in conversation:", len(final_state.values["messages"]))

# Display the conversation flow
ChatPromptTemplate.from_messages(final_state.values["messages"]).pretty_print()

# Building a Reflexion Agent


In [2]:
# Construct tools

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper

# Initialize search tool
web_search = TavilySearchAPIWrapper()
tavily_tool = TavilySearchResults(api_wrapper=web_search, max_results=5)

# Agent prompt template
actor_prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert technical educator specializing in machine learning and neural networks.
                Current time: {time}
                1. {primary_instruction}
                2. Reflect and critique your answer. Be severe to maximize improvement.
                3. Recommend search queries to research information and improve your answer.""",
        ),
        MessagesPlaceholder(variable_name="messages"),
        (
            "user",
            "\n\n<s>Reflect on the user's original question and the"
            " actions taken thus far. Respond using the {function_name} function.</reminder>",
        ),
    ]
).partial(
    time=lambda: datetime.datetime.now().isoformat(),
)

# Pydantic models for structured output
class Reflection(BaseModel):
    missing: str = Field(description="Critique of what is missing.")
    superfluous: str = Field(description="Critique of what is superfluous")

class GenerateResponse(BaseModel):
    """Generate response. Provide an answer, critique, and then follow up with search queries to improve the answer."""

    response: str = Field(description="~250 word detailed answer to the question.")
    reflection: Reflection = Field(description="Your reflection on the initial answer.")
    research_queries: list[str] = Field(
        description="1-3 search queries for researching improvements to address the critique of your current answer."
    )

# Agent with retry logic
class AdaptiveResponder:
    def __init__(self, chain, output_parser):
        self.chain = chain
        self.output_parser = output_parser

    def generate(self, conversation_state: dict):
        llm_response = None
        for retry_count in range(3):
            llm_response = self.chain.invoke(
                {"messages": conversation_state["messages"]}, {"tags": [f"attempt:{retry_count}"]}
            )
            try:
                self.output_parser.invoke(llm_response)
                return {"messages": llm_response}
            except ValidationError as validation_error:
                # Fix: Convert schema dict to JSON string
                schema_json = json.dumps(self.output_parser.model_json_schema(), indent=2)
                conversation_state = conversation_state + [
                    llm_response,
                    ToolMessage(
                        content=f"{repr(validation_error)}\n\nPay close attention to the function schema.\n\n{schema_json}\n\nRespond by fixing all validation errors.",
                        tool_call_id=llm_response.tool_calls[0]["id"],
                    ),
                ]
        return {"messages": llm_response}

# Initial answer chain
initial_response_chain = actor_prompt_template.partial(
    primary_instruction="Provide a detailed ~250 word explanation suitable for someone with basic programming background.",
    function_name=GenerateResponse.__name__,
) | llm.bind_tools(tools=[GenerateResponse])

response_parser = PydanticToolsParser(tools=[GenerateResponse])

initial_responder = AdaptiveResponder(
    chain=initial_response_chain, output_parser=response_parser
)

example_question = "What is the difference between supervised and unsupervised learning?"
initial = initial_responder.generate(
    {"messages": [HumanMessage(content=example_question)]}
)

initial

# Revision instructions
improvement_guidelines = """Revise your previous explanation using the new information.
    - You should use the previous critique to add important technical details to your explanation.
    - You MUST include numerical citations in your revised answer to ensure it can be verified.
    - Add a "References" section to the bottom of your answer (which does not count towards the word limit).
    - For the references field, provide a clean list of URLs only (e.g., ["https://example.com", "https://example2.com"])
    - You should use the previous critique to remove superfluous information from your answer and make SURE it is not more than 250 words.
    - Keep the explanation accessible for someone with basic programming background while being technically accurate.
"""

class ImproveResponse(GenerateResponse):
    """Improve your original answer to your question. Provide an answer, reflection,
    cite your reflection with references, and finally
    add search queries to improve the answer."""

    sources: list[str] = Field(
        description="List of reference URLs that support your answer. Each reference should be a clean URL string."
    )

# Revision chain
improvement_chain = actor_prompt_template.partial(
    primary_instruction=improvement_guidelines,
    function_name=ImproveResponse.__name__,
) | llm.bind_tools(tools=[ImproveResponse])


improvement_parser = PydanticToolsParser(tools=[ImproveResponse])
response_improver = AdaptiveResponder(chain=improvement_chain, output_parser=improvement_parser)

revised = response_improver.generate(
    {
        "messages": [
            HumanMessage(content=example_question),
            initial["messages"],
            ToolMessage(
                tool_call_id=initial["messages"].tool_calls[0]["id"],
                content=json.dumps(
                    tavily_tool.invoke(
                        {
                            "query": initial["messages"].tool_calls[0]["args"][
                                "research_queries"
                            ][0]
                        }
                    )
                ),
            ),
        ]
    }
)

revised["messages"]

# Tool execution function
def execute_search_queries(research_queries: list[str], **kwargs):
    """Execute the generated search queries."""
    return tavily_tool.batch([{"query": search_term} for search_term in research_queries])

# Tool node
search_executor = ToolNode(
    [
        StructuredTool.from_function(execute_search_queries, name=GenerateResponse.__name__),
        StructuredTool.from_function(execute_search_queries, name=ImproveResponse.__name__),
    ]
)

# Graph state definition
class State(TypedDict):
    messages: Annotated[list, add_messages]

# Helper functions for looping logic
def get_iteration_count(message_history: list):
    """ Counts backwards through messages until it hits a non-tool, non-AI message
    This helps determine how many tool execution cycles have occurred recently"""

    iteration_count = 0
    # Iterate through messages in reverse order (most recent first)
    for message in message_history[::-1]:
        if message.type not in {"tool", "ai"}:
            break
        iteration_count += 1
    return iteration_count

def determine_next_action(state: list):
    """
    Conditional edge function that determines whether to continue the loop or end.

    Args:
        state: Current workflow state containing messages

    Returns:
        str: Next node to execute ("search_and_research") or END to terminate

    Logic:
    - Counts recent iterations using get_iteration_count()
    - If we've exceeded MAXIMUM_CYCLES, stop the workflow
    - Otherwise, continue with another tool execution cycle
    """
    # in our case, we'll just stop after N plans
    current_iterations = get_iteration_count(state["messages"])
    if current_iterations > MAXIMUM_CYCLES:
        return END
    return "search_and_research"


# Graph construction
MAXIMUM_CYCLES = 5
workflow_builder = StateGraph(State)

# Add nodes
workflow_builder.add_node("create_draft", initial_responder.generate)
workflow_builder.add_node("search_and_research", search_executor)
workflow_builder.add_node("enhance_response", response_improver.generate)

# Add edges
workflow_builder.add_edge(START, "create_draft")
workflow_builder.add_edge("create_draft", "search_and_research")
workflow_builder.add_edge("search_and_research", "enhance_response")

# Add conditional edges for looping
workflow_builder.add_conditional_edges("enhance_response", determine_next_action, ["search_and_research", END])

# Compile the graph
reflexion_workflow = workflow_builder.compile()

from IPython.display import Image, display

# Show the agent
display(Image(reflexion_workflow.get_graph().draw_png()))

# Run the agent with the neural networks question
target_question = "How do neural networks actually learn?"

print(f"Running Reflexion agent with question: {target_question}")
print("=" * 60)

events = reflexion_workflow.stream(
    {"messages": [("user", target_question)]},
    stream_mode="values",
)

for i, step in enumerate(events):
    print(f"\nStep {i}")
    print("-" * 40)
    step["messages"][-1].pretty_print()